# Homework 01

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

# load_dotenv(dotenv_path=os.path.join(os.path.dirname(__file__), '../../.env'))
load_dotenv()

openai_client = OpenAI()

## Q1: How many lesson pages

In [10]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Lessons pages loaded: {len(documents)}")

Lessons pages loaded: 72


## Q2: Indexing and searching

In [11]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query, boost_dict={"content": 1.0}, num_results=5)
if search_results:
    print(f"First result (filename): {search_results[0]['filename']}")
else:
    print("No results found.")

First result (filename): 01-agentic-rag/lessons/14-agentic-loop.md


## Q3: RAG

In [12]:
from rag_helper import RAGBase

assistant = RAGBase(index=index, llm_client=openai_client, model="gpt-5.4-mini")
answer, usage = assistant.rag(query)
print(f"Input tokens (Q3 without chunks): {usage.input_tokens}")
q3_tokens = usage.input_tokens

Input tokens (Q3 without chunks): 7136


## Q4: Chunking

In [13]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Chunks: {len(chunks)}")

Chunks: 295


## Q5

In [14]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

chunk_assistant = RAGBase(
    index=chunk_index, llm_client=openai_client, model="gpt-5.4-mini"
)
answer_chunked, usage_chunked = chunk_assistant.rag(query)
print(f"Input tokens (Q5 with chunks): {usage_chunked.input_tokens}")
print(
    f"{q3_tokens} / {usage_chunked.input_tokens} = ~{q3_tokens / usage_chunked.input_tokens:.2f}x"
)

Input tokens (Q5 with chunks): 2319
7136 / 2319 = ~3.08x


## Q6: Agent

In [20]:
def search_tool(query: str) -> list[dict]:
    """
    Search the course lessons database for entries matching the given query.
    """
    search_tool.calls += 1  # counter
    return chunk_index.search(query, boost_dict={"content": 1.0}, num_results=5)


search_tool.calls = 0

toyai_client = OpenAIClient(client=openai_client, model="gpt-5.4-mini")
tools = Tools()
tools.add_tool(search_tool)

instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""
question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

runner = OpenAIResponsesRunner(
    llm_client=toyai_client, tools=tools, developer_prompt=instructions
)

# loop_result = runner.loop(prompt=question)

# final_text = loop_result.last_message

# print(f"Search calls: {search_tool.calls}")
# print(f"Agent answer: {final_text}")

max_iterations = 5
current_messages = list(messages)

for i in range(max_iterations):
    loop_result = runner.loop(prompt=question, previous_messages=current_messages)
    current_messages.extend(loop_result.new_messages)

    # Если на последнем шаге не было вызовов функций, значит агент закончил сам
    # Проверяем, есть ли вызовы функций в добавленных сообщениях
    has_tool_calls = any(
        getattr(msg, "tool_calls", None)
        for msg in loop_result.new_messages
        if hasattr(msg, "role") and msg.role == "assistant"
    )
    if not has_tool_calls:
        break

final_text = loop_result.last_message

In [21]:
print(f"Search calls: {search_tool.calls}")
print(
    f"Used tokens: {loop_result.tokens.input_tokens} input, {loop_result.tokens.output_tokens} output"
)
if loop_result.cost:
    print(f"Cost: ${loop_result.cost.total_cost:.6f}")
print(f"Agent answer:\n{final_text}")

Search calls: 4
Used tokens: 10713 input, 556 output
Cost: $0.010537
Agent answer:
The course describes the **agentic loop** as a `while`-style cycle:

1. Send the user request to the LLM
2. If the LLM returns a tool call, execute that tool
3. Send the tool result back to the LLM
4. Repeat until the model returns a final answer with no more tool calls

In other words, the model is not just answering once — it can **decide what to do next** based on the intermediate results. The lesson says this is the foundation of every agent framework, and that frameworks just manage this loop for you.

### How that differs from plain RAG

**Plain RAG** is a simpler 3-step pipeline:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

So plain RAG does:

- one retrieval step
- one prompt-building step
- one LLM call

The retrieval happens **outside** the model. Your code decides what to search, fe